# SMPS Full Validation Notebook

This notebook provides a complete environment for running the Soil Moisture Prediction System (SMPS) model validation. It combines physics-based modeling with machine learning to predict soil moisture across multiple depths and forecast horizons.

## Overview
- **Physics Model**: Simple Water Balance model with pedotransfer functions
- **ML Model**: Hybrid ensemble model with uncertainty quantification
- **Data Sources**: ISMN observations, satellite data (GEE), weather (Open-Meteo), soil properties
- **Validation**: Multi-depth, multi-horizon evaluation with quality filtering

## Requirements
- Python 3.9+
- Google Earth Engine authentication (for satellite data)
- ISMN data directory
- Sufficient computational resources (RAM: 16GB+, Disk: 50GB+)

---
**Last Updated**: January 29, 2026
**SMPS Version**: 0.1.0

## 1. Environment Setup

Install and verify all required dependencies for the SMPS system.

In [37]:
# Environment Setup Cell
import sys
import os
from pathlib import Path

# Detect if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally")

if IN_COLAB:
    # Colab environment setup
    project_root = Path("/content/smps")
    print("Note: Make sure to clone/upload the SMPS repository to /content/smps")
else:
    # Local environment
    project_root = Path("/home/viv/SMPS")

# Add the src directory to Python path
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Python path includes: {src_path}")
print(f"Python version: {sys.version}")

# Verify key directories exist
required_dirs = ["data", "results", "scripts", "src/smps"]
missing_dirs = []
for dir_name in required_dirs:
    dir_path = project_root / dir_name
    if dir_path.exists():
        print(f"✓ {dir_name}: {dir_path}")
    else:
        print(f"✗ {dir_name}: {dir_path} (MISSING)")
        missing_dirs.append(dir_name)

if missing_dirs:
    if IN_COLAB:
        print("\n⚠ Some directories are missing. Make sure the SMPS repository is properly cloned.")
        print("You can clone it with: !git clone https://github.com/your-username/smps.git /content/smps")
    else:
        print("\n⚠ Some directories are missing. Make sure you're in the correct SMPS project directory.")

# Check for virtual environment (only relevant for local execution)
if not IN_COLAB:
    venv_path = Path("/home/viv/SMPS/.venv")
    if venv_path.exists():
        print(f"✓ Virtual environment found: {venv_path}")
    else:
        print("! No virtual environment detected - dependencies will be installed in the notebook")

✓ Running in Google Colab
Note: Make sure to clone/upload the SMPS repository to /content/smps
Project root: /content/smps
Python path includes: /content/smps/src
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
✓ data: /content/smps/data
✓ results: /content/smps/results
✓ scripts: /content/smps/scripts
✓ src/smps: /content/smps/src/smps


In [16]:
# Clone SMPS Repository (Colab only)
if IN_COLAB:
    import os
    if not os.path.exists('/content/smps'):
        print("Cloning SMPS repository...")
        !git clone https://github.com/viv-mwirigi/SMPS.git /content/smps
        print("✓ Repository cloned successfully")
    else:
        print("✓ SMPS repository already exists")

    # Change to the repository directory
    os.chdir('/content/smps')
    print(f"Changed working directory to: {os.getcwd()}")

Cloning SMPS repository...
Cloning into '/content/smps'...
remote: Enumerating objects: 1518, done.
remote: Counting objects: 100% (2/2), done.
remote: Total 1518 (delta 1), reused 1 (delta 1), pack-reused 1516 (from 1)
Receiving objects: 100% (1518/1518), 73.22 MiB | 20.14 MiB/s, done.
Resolving deltas: 100% (903/903), done.
✓ Repository cloned successfully
Changed working directory to: /content/smps


In [35]:
# Copy data from local workspace to Colab (if available)
if IN_COLAB:
    import shutil
    local_data_path = "/home/viv/SMPS/data"
    colab_data_path = "/content/smps/data"

    if os.path.exists(local_data_path):
        print("Copying data from local workspace to Colab...")
        try:
            shutil.copytree(local_data_path, colab_data_path, dirs_exist_ok=True)
            print("✓ Data copied successfully")
        except Exception as e:
            print(f"⚠ Could not copy data: {e}")
            print("You may need to upload the data manually to /content/smps/data/")
    else:
        print("⚠ Local data not found in Colab environment")
        print("\n📁 To add your data to Colab:")
        print("1. Click the folder icon in the left sidebar")
        print("2. Right-click and select 'Upload' or drag & drop your data folder")
        print("3. Upload the 'data' folder from your SMPS project")
        print("4. Or mount Google Drive and copy from there")
        print("\nAlternatively, you can run this notebook locally where the data is available.")

⚠ Local data not found in Colab environment

📁 To add your data to Colab:
1. Click the folder icon in the left sidebar
2. Right-click and select 'Upload' or drag & drop your data folder
3. Upload the 'data' folder from your SMPS project
4. Or mount Google Drive and copy from there

Alternatively, you can run this notebook locally where the data is available.


In [34]:
# Debug Environment
import os
print(f"Current working directory: {os.getcwd()}")
print(f"Files in current directory: {os.listdir('.')}")
print(f"Python executable: {sys.executable}")
print(f"Python path: {sys.path[:3]}")  # Show first 3 paths

# Check if we can access the SMPS directory
smps_path = "/home/viv/SMPS"
if os.path.exists(smps_path):
    print(f"✓ SMPS directory exists: {smps_path}")
    print(f"Contents: {os.listdir(smps_path)}")
else:
    print(f"✗ SMPS directory not found: {smps_path}")

# Try to import a basic module
try:
    import pandas as pd
    print("✓ pandas imported successfully")
except ImportError as e:
    print(f"✗ pandas import failed: {e}")

Current working directory: /content/smps
Files in current directory: ['src', 'data', '0.95', 'tests', 'results', 'Makefile', '.gitignore', 'pyproject.toml', 'mlflow.db', 'scripts', '.git']
Python executable: /usr/bin/python3
Python path: ['/content/smps/src', '/content/src', '/home/viv/SMPS/src']
✗ SMPS directory not found: /home/viv/SMPS
✓ pandas imported successfully


## 2. Import Libraries

Import all necessary libraries and SMPS modules.

In [24]:
# Import Libraries
import logging
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
import json
import time

import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure logging for notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("smps.notebook")

print("✓ Standard libraries imported")

# SMPS Core Imports
try:
    from smps.data.sources.ismn_loader import ISMNStationLoader, ISMNStationData, get_daily_soil_moisture
    from smps.physics.simple_water_balance import (
        SimpleWaterBalance,
        create_simple_config_improved,
        create_simple_config
    )
    from smps.physics.pedotransfer import estimate_soil_parameters_tropical, TropicalSoilCorrections
    from smps.validation.physics_metrics import run_physics_validation
    from smps.data.sources.weather import OpenMeteoSource
    from smps.data.sources.gee_satellite import GoogleEarthEngineSatelliteSource
    from smps.data.sources.base import DataFetchRequest
    print("✓ SMPS core modules imported")
except ImportError as e:
    print(f"✗ Failed to import SMPS core modules: {e}")
    print("Ensure the src directory is in Python path")

# ML Imports
try:
    from smps.ml.hybrid_model import (
        HybridSoilMoistureModel,
        ResidualLearner,
        PhysicsResidualTarget,
        ResidualLearnerConfig
    )
    from smps.ml.enhanced_hybrid_model import (
        UncertaintyAwareHybridModel,
        EnhancedResidualLearnerConfig,
        UncertaintyConfig,
        ResidualQualityAssessment
    )
    from smps.ml.domain_shift_detection import (
        DomainShiftConfig,
        DomainShiftAwareModel
    )
    from smps.ml.ensemble import StackingEnsemble, EnsembleConfig, BaseModelConfig
    from smps.ml.spatiotemporal_features import SpatialFeatureEngineer, SpatialConfig
    from smps.ml.validation import DataSplitter, SplitConfig
    from smps.ml.trainer import MLTrainingPipeline
    print("✓ SMPS ML modules imported")
except ImportError as e:
    print(f"✗ Failed to import SMPS ML modules: {e}")

print("\n✓ All imports completed successfully!")

✓ Standard libraries imported
✓ SMPS core modules imported
✓ SMPS ML modules imported

✓ All imports completed successfully!


In [23]:
# Install Missing Dependencies (Colab)
if IN_COLAB:
    print("Installing missing dependencies...")
    !pip install -q optuna scikit-learn lightgbm xgboost seaborn earthengine-api mlflow
    print("✓ Dependencies installed")

# Test key ML imports
try:
    import optuna
    import sklearn
    import lightgbm
    import xgboost
    import seaborn
    import mlflow
    print("✓ All ML dependencies available")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")

Installing missing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.5/788.5 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 22.7 MB/s eta 0:00:00
✓ Dependencies installed
✓ All ML dependencies available


## 3. Configure Paths and Settings

Set up all paths, configurations, and runtime parameters.

In [28]:
# Configuration Cell

# ===== PATH CONFIGURATION =====
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

# ISMN Data Directory (choose one of the available datasets)
ISMN_DATA_DIR = DATA_DIR / "ismn" / "Data_separate_files_header_20170105_20250105_12892_F2PyW_20260105"
# Alternative: ISMN_DATA_DIR = DATA_DIR / "ismn" / "Data_separate_files_header_20170105_20250105_12892_gdf6E_20260105"

# Output directory for this run
OUTPUT_DIR = RESULTS_DIR / f"notebook_validation_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"ISMN data: {ISMN_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# ===== VALIDATION CONFIGURATION =====
# Date range for validation
START_DATE = "2019-01-01"
END_DATE = "2021-12-31"

# Networks to process (None = all networks)
NETWORKS = None  # or ["AMMA-CATCH", "TAHMO"] for specific networks

# Maximum number of stations to process (None = all)
MAX_STATIONS = 10  # Set to None for full run, 10 for testing

# Forecast horizons for evaluation (in days)
HORIZONS = {
    '0h': 0,    # Nowcast (same day)
    '24h': 1,   # 1-day ahead
    '72h': 3,   # 3-day ahead
    '168h': 7   # 7-day ahead (weekly)
}

# Station quality thresholds for filtering
STATION_QUALITY_THRESHOLDS = {
    'max_abs_bias': 0.12,      # Max absolute bias |obs_mean - physics_mean|
    'min_kge': -0.5,           # Min acceptable KGE (allow some negative for correction)
    'max_obs_mean_deviation': 0.05,  # Max deviation from realistic soil moisture (0.1-0.5)
}

# ===== MODEL CONFIGURATION =====
# Enable quality filtering
ENABLE_QUALITY_FILTERING = True

# Spatiotemporal feature configuration
SPATIAL_CONFIG = SpatialConfig(
    max_neighbor_distance_km=50.0,
    min_neighbors=2,
    regional_radius_km=100.0
)

# ML Model configuration
ENSEMBLE_CONFIG = EnsembleConfig(
    base_models=['randomforest', 'xgboost', 'lightgbm'],
    meta_model='ridge',
    n_folds=3
)

# Uncertainty configuration
UNCERTAINTY_CONFIG = UncertaintyConfig(
    use_ensemble=True,
    ensemble_size=5,  # Reduced for Colab
    use_quantile_regression=True
)

print("\n✓ Configuration loaded successfully!")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Max stations: {MAX_STATIONS}")
print(f"Quality filtering: {'Enabled' if ENABLE_QUALITY_FILTERING else 'Disabled'}")

Project root: /content
Data directory: /content/data
Results directory: /content/results
ISMN data: /content/data/ismn/Data_separate_files_header_20170105_20250105_12892_F2PyW_20260105
Output directory: /content/results/notebook_validation_20260129_074158

✓ Configuration loaded successfully!
Date range: 2019-01-01 to 2021-12-31
Max stations: 10
Quality filtering: Enabled


## 4. Load Data Sources

Initialize and verify all data sources required for the validation.

In [36]:
# Data Sources Initialization

print("Initializing data sources...")

# ===== ISMN STATION LOADER =====
try:
    ismn_loader = ISMNStationLoader(ISMN_DATA_DIR)
    print("✓ ISMN loader initialized")
except Exception as e:
    print(f"✗ Failed to initialize ISMN loader: {e}")
    raise

# ===== WEATHER SOURCE =====
try:
    weather_source = OpenMeteoSource(cache_dir=OUTPUT_DIR / "cache" / "weather")
    print("✓ Weather source (Open-Meteo) initialized")
except Exception as e:
    print(f"✗ Failed to initialize weather source: {e}")
    weather_source = None

# ===== SATELLITE SOURCE (GOOGLE EARTH ENGINE) =====
try:
    satellite_source = GoogleEarthEngineSatelliteSource()
    has_gee = True
    print("✓ Satellite source (Google Earth Engine) initialized")
except Exception as e:
    print(f"⚠ Google Earth Engine not available: {e}")
    print("  Satellite features will be skipped")
    satellite_source = None
    has_gee = False

# ===== SPATIOTEMPORAL FEATURE ENGINEER =====
try:
    spatial_engineer = SpatialFeatureEngineer(SPATIAL_CONFIG)
    print("✓ Spatiotemporal feature engineer initialized")
except Exception as e:
    print(f"✗ Failed to initialize spatial engineer: {e}")
    spatial_engineer = None

print("\nData sources summary:")
print(f"  ISMN Data: {ISMN_DATA_DIR.exists()}")
print(f"  Weather: {'Available' if weather_source else 'Unavailable'}")
print(f"  Satellite: {'Available' if has_gee else 'Unavailable'}")
print(f"  Spatial Features: {'Available' if spatial_engineer else 'Unavailable'}")

Initializing data sources...
✓ ISMN loader initialized
✓ Weather source (Open-Meteo) initialized
⚠ Google Earth Engine not available: Failed to initialize Earth Engine: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.. Make sure you have authenticated with 'earthengine authenticate' and have a Google Cloud project with Earth Engine enabled.
  Satellite features will be skipped
✓ Spatiotemporal feature engineer initialized

Data sources summary:
  ISMN Data: False
  Weather: Available
  Satellite: Unavailable
  Spatial Features: Available


## 5. Build Dataset

Load ISMN stations and construct the complete feature dataset.

In [31]:
# Dataset Building

def load_stations(networks=None, max_stations=None):
    """Load ISMN stations with filtering."""
    print("Loading ISMN stations...")

    if networks:
        station_data_list = []
        for network in networks:
            try:
                stations_dict = ismn_loader.load_network(network)
                station_data_list.extend(stations_dict.values())
                print(f"  Loaded {len(stations_dict)} stations from {network}")
            except Exception as e:
                print(f"  Failed to load {network}: {e}")
    else:
        try:
            stations_dict = ismn_loader.load_all_stations()
            station_data_list = list(stations_dict.values())
            print(f"  Loaded {len(station_data_list)} stations from all networks")
        except Exception as e:
            print(f"  Failed to load stations: {e}")
            return []

    # Filter to stations with data in date range
    start_dt = pd.to_datetime(START_DATE)
    end_dt = pd.to_datetime(END_DATE)

    valid_stations = []
    for station in station_data_list:
        if station.daily_data is not None:
            df = station.daily_data
            df_filtered = df[(df['date'] >= start_dt) & (df['date'] <= end_dt)]
            if len(df_filtered) >= 100:  # At least 100 days of data
                valid_stations.append(station)

    if max_stations:
        valid_stations = valid_stations[:max_stations]

    print(f"✓ Filtered to {len(valid_stations)} valid stations")
    return valid_stations

def process_station_features(station_data):
    """Process a single station to extract features and physics predictions."""
    try:
        station_id = station_data.station_id
        print(f"  Processing {station_id}...")

        # Get daily soil moisture data
        daily_data = station_data.daily_data
        if daily_data is None or len(daily_data) < 50:
            return None

        # Filter date range
        start_dt = pd.to_datetime(START_DATE)
        end_dt = pd.to_datetime(END_DATE)
        daily_data = daily_data[(daily_data['date'] >= start_dt) & (daily_data['date'] <= end_dt)]

        if len(daily_data) < 50:
            return None

        # Initialize physics model
        lat, lon = station_data.latitude, station_data.longitude

        # Get soil parameters
        try:
            soil_params = estimate_soil_parameters_tropical(
                station_data.sand_fraction,
                station_data.clay_fraction,
                station_data.bulk_density
            )
        except:
            # Fallback soil parameters
            soil_params = {
                'theta_sat': 0.45,
                'theta_res': 0.05,
                'alpha': 0.02,
                'n': 1.5,
                'ksat': 0.001
            }

        # Create physics model configuration
        config = create_simple_config_improved(
            latitude=lat,
            longitude=lon,
            soil_params=soil_params,
            root_depth=1.0  # Default
        )

        physics_model = SimpleWaterBalance(config)

        # Get weather data
        weather_data = None
        if weather_source:
            try:
                weather_request = DataFetchRequest(
                    latitude=lat,
                    longitude=lon,
                    start_date=start_dt,
                    end_date=end_dt
                )
                weather_data = weather_source.fetch(weather_request)
            except Exception as e:
                print(f"    Weather fetch failed for {station_id}: {e}")

        # Get satellite data
        satellite_data = None
        if has_gee and satellite_source:
            try:
                satellite_request = DataFetchRequest(
                    latitude=lat,
                    longitude=lon,
                    start_date=start_dt,
                    end_date=end_dt
                )
                satellite_data = satellite_source.fetch(satellite_request)
            except Exception as e:
                print(f"    Satellite fetch failed for {station_id}: {e}")

        # Build feature dataframe
        features_list = []

        for idx, row in daily_data.iterrows():
            date = row['date']
            obs_sm = row['soil_moisture']

            # Skip if observation is invalid
            if pd.isna(obs_sm) or obs_sm < 0 or obs_sm > 1:
                continue

            feature_row = {
                'station_id': station_id,
                'date': date,
                'latitude': lat,
                'longitude': lon,
                'observed_soil_moisture': obs_sm,
                'depth': getattr(station_data, 'depth', 0.05),  # Default 5cm
            }

            # Add weather features
            if weather_data is not None:
                weather_row = weather_data[weather_data['date'] == date]
                if not weather_row.empty:
                    wrow = weather_row.iloc[0]
                    feature_row.update({
                        'precipitation': wrow.get('precipitation', 0),
                        'temperature': wrow.get('temperature_2m', 25),
                        'humidity': wrow.get('relative_humidity_2m', 70),
                        'wind_speed': wrow.get('wind_speed_10m', 2),
                        'et0': wrow.get('et0', 3),
                    })

            # Add satellite features
            if satellite_data is not None:
                sat_row = satellite_data[satellite_data['date'] == date]
                if not sat_row.empty:
                    srow = sat_row.iloc[0]
                    feature_row.update({
                        'ndvi': srow.get('ndvi', 0.3),
                        'evi': srow.get('evi', 0.3),
                        'lai': srow.get('lai', 0.5),
                    })

            # Add soil parameters
            feature_row.update({
                'sand_fraction': getattr(station_data, 'sand_fraction', 0.4),
                'clay_fraction': getattr(station_data, 'clay_fraction', 0.3),
                'bulk_density': getattr(station_data, 'bulk_density', 1.3),
                'theta_sat': soil_params['theta_sat'],
                'theta_res': soil_params['theta_res'],
                'alpha': soil_params['alpha'],
                'n': soil_params['n'],
                'ksat': soil_params['ksat'],
            })

            # Run physics model for different horizons
            for horizon_name, horizon_days in HORIZONS.items():
                try:
                    pred_date = date + timedelta(days=horizon_days)
                    physics_pred = physics_model.predict(pred_date)

                    if physics_pred is not None:
                        feature_row[f'physics_pred_{horizon_name}'] = physics_pred
                    else:
                        feature_row[f'physics_pred_{horizon_name}'] = np.nan
                except:
                    feature_row[f'physics_pred_{horizon_name}'] = np.nan

            features_list.append(feature_row)

        if features_list:
            station_df = pd.DataFrame(features_list)
            print(f"    ✓ Generated {len(station_df)} feature rows for {station_id}")
            return station_df
        else:
            print(f"    ✗ No valid features generated for {station_id}")
            return None

    except Exception as e:
        print(f"    ✗ Failed to process {station_data.station_id}: {e}")
        return None

# ===== EXECUTE DATASET BUILDING =====
print("Building complete feature dataset...")
print("=" * 50)

# Load stations
valid_stations = load_stations(networks=NETWORKS, max_stations=MAX_STATIONS)

if not valid_stations:
    raise ValueError("No valid stations found!")

# Process each station
all_station_features = []
for station_data in tqdm(valid_stations, desc="Processing stations"):
    features_df = process_station_features(station_data)
    if features_df is not None and len(features_df) >= 50:
        all_station_features.append(features_df)

if not all_station_features:
    raise ValueError("No station feature data collected!")

# Combine all station data
combined_df = pd.concat(all_station_features, ignore_index=True)
print(f"\n✓ Combined dataset: {len(combined_df)} rows, {len(combined_df.columns)} columns")

# Add spatiotemporal features
if spatial_engineer:
    print("Adding spatiotemporal features...")
    try:
        # Build site metadata
        site_metadata = {}
        for station_data in valid_stations:
            site_metadata[station_data.station_id] = {
                'latitude': station_data.latitude,
                'longitude': station_data.longitude,
                'elevation_m': getattr(station_data, 'elevation_m', 200.0),
                'slope_percent': 5.0,
                'land_use_code': 1.0,
            }

        # Add spatiotemporal features
        combined_df = spatial_engineer.engineer_spatiotemporal_features(
            combined_df, site_metadata
        )
        print(f"✓ Spatiotemporal features added: {len(combined_df)} rows, {len(combined_df.columns)} columns")

    except Exception as e:
        print(f"⚠ Spatiotemporal feature engineering failed: {e}")

# Save combined dataset
combined_df.to_csv(OUTPUT_DIR / "combined_features.csv", index=False)
print(f"✓ Dataset saved to {OUTPUT_DIR / 'combined_features.csv'}")

# Display dataset summary
print("\nDataset Summary:")
print(f"  Stations: {combined_df['station_id'].nunique()}")
print(f"  Date range: {combined_df['date'].min()} to {combined_df['date'].max()}")
print(f"  Total observations: {len(combined_df)}")
print(f"  Features: {len(combined_df.columns)}")
print(f"  Missing values: {combined_df.isnull().sum().sum()}")

# Show first few rows
combined_df.head()

Building complete feature dataset...
Loading ISMN stations...
  Failed to load stations: No ISMN datasets found


ValueError: No valid stations found!

## 6. Run Physics Model Calibration

Calibrate physics model parameters for each station.

In [ ]:
# Physics Model Calibration

def assess_station_quality(station_id, station_df):
    """Assess the quality of physics predictions for a station."""
    quality = {
        'station_id': station_id,
        'include': True,
        'reasons': [],
        'metrics': {}
    }

    # Get physics predictions and observations
    obs_col = 'observed_soil_moisture'
    physics_cols = [col for col in station_df.columns if col.startswith('physics_pred_')]

    if not physics_cols:
        quality['include'] = False
        quality['reasons'].append("No physics predictions available")
        return quality

    # Use the nowcast prediction for quality assessment
    physics_col = 'physics_pred_0h'
    if physics_col not in station_df.columns:
        physics_col = physics_cols[0]  # Use first available

    valid_data = station_df[[obs_col, physics_col]].dropna()

    if len(valid_data) < 30:
        quality['include'] = False
        quality['reasons'].append(f"Insufficient data: {len(valid_data)} points")
        return quality

    obs = valid_data[obs_col].values
    pred = valid_data[physics_col].values

    # Calculate basic metrics
    bias = np.mean(pred - obs)
    abs_bias = abs(bias)
    mae = np.mean(np.abs(pred - obs))
    rmse = np.sqrt(np.mean((pred - obs)**2))

    # Kling-Gupta Efficiency
    obs_mean = np.mean(obs)
    pred_mean = np.mean(pred)
    obs_std = np.std(obs)
    pred_std = np.std(pred)

    r = np.corrcoef(obs, pred)[0, 1] if len(obs) > 1 else 0

    alpha = pred_std / obs_std if obs_std > 0 else 0
    beta = pred_mean / obs_mean if obs_mean > 0 else 0

    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

    quality['metrics'] = {
        'bias': bias,
        'abs_bias': abs_bias,
        'mae': mae,
        'rmse': rmse,
        'r': r,
        'kge': kge,
        'n_points': len(valid_data)
    }

    # Apply quality thresholds
    if abs_bias > STATION_QUALITY_THRESHOLDS['max_abs_bias']:
        quality['include'] = False
        quality['reasons'].append(".3f")

    if kge < STATION_QUALITY_THRESHOLDS['min_kge']:
        quality['include'] = False
        quality['reasons'].append(".3f")

    obs_mean_val = np.mean(obs)
    if abs(obs_mean_val - 0.3) > STATION_QUALITY_THRESHOLDS['max_obs_mean_deviation']:
        quality['include'] = False
        quality['reasons'].append(".3f")

    return quality

# ===== EXECUTE QUALITY ASSESSMENT =====
if ENABLE_QUALITY_FILTERING:
    print("Performing station quality assessment...")
    print("=" * 50)

    station_ids = combined_df['station_id'].unique()
    included_stations = []
    excluded_stations = []
    station_qualities = {}

    for station_id in tqdm(station_ids, desc="Assessing stations"):
        station_df = combined_df[combined_df['station_id'] == station_id]
        quality = assess_station_quality(station_id, station_df)
        station_qualities[station_id] = quality

        if quality['include']:
            included_stations.append(station_id)
        else:
            excluded_stations.append(station_id)
            print(f"  Excluding {station_id}: {', '.join(quality['reasons'])}")

    print("
Quality filtering results:")
    print(f"  Included: {len(included_stations)} stations")
    print(f"  Excluded: {len(excluded_stations)} stations")

    if excluded_stations:
        print("\nExcluded stations:")
        for s in excluded_stations:
            q = station_qualities[s]
            print(f"    {s}: {q['reasons']}")

    # Filter the combined dataframe
    original_len = len(combined_df)
    combined_df = combined_df[combined_df['station_id'].isin(included_stations)].copy()
    print(f"\nFiltered dataset: {len(combined_df)} rows (removed {original_len - len(combined_df)})")

    if len(combined_df) < 100:
        raise ValueError("Insufficient data after quality filtering!")

else:
    print("Quality filtering disabled - using all stations")

# Save filtered dataset
combined_df.to_csv(OUTPUT_DIR / "combined_features_filtered.csv", index=False)
print(f"✓ Filtered dataset saved to {OUTPUT_DIR / 'combined_features_filtered.csv'}")

# Display quality summary
if ENABLE_QUALITY_FILTERING:
    quality_df = pd.DataFrame([
        {**q['metrics'], 'station_id': sid, 'included': q['include']}
        for sid, q in station_qualities.items()
    ])
    print("\nQuality Metrics Summary:")
    print(quality_df.describe())

## 7. Train Machine Learning Model

Train the hybrid ML model with ensemble methods and uncertainty quantification.

In [ ]:
# Machine Learning Model Training

def prepare_ml_data(df, target_horizon='0h'):
    """Prepare data for ML training with physics residuals as targets."""
    print(f"Preparing ML data for horizon {target_horizon}...")

    # Select features
    base_features = [
        'latitude', 'longitude', 'depth',
        'precipitation', 'temperature', 'humidity', 'wind_speed', 'et0',
        'sand_fraction', 'clay_fraction', 'bulk_density',
        'theta_sat', 'theta_res', 'alpha', 'n', 'ksat'
    ]

    # Add satellite features if available
    satellite_features = []
    if 'ndvi' in df.columns:
        satellite_features.extend(['ndvi', 'evi', 'lai'])
    if any(col.startswith('spatial_') for col in df.columns):
        spatial_features = [col for col in df.columns if col.startswith('spatial_')]
        base_features.extend(spatial_features)

    feature_cols = base_features + satellite_features

    # Get target (physics residual)
    physics_col = f'physics_pred_{target_horizon}'
    target_col = 'observed_soil_moisture'

    # Filter valid data
    valid_df = df[feature_cols + [physics_col, target_col, 'station_id', 'date']].dropna()

    if len(valid_df) < 100:
        print(f"  ⚠ Insufficient data for {target_horizon}: {len(valid_df)} points")
        return None, None

    # Calculate residual target
    valid_df = valid_df.copy()
    valid_df['residual_target'] = valid_df[target_col] - valid_df[physics_col]

    # Prepare features and target
    X = valid_df[feature_cols]
    y = valid_df['residual_target']
    metadata = valid_df[['station_id', 'date', physics_col, target_col]]

    print(f"  ✓ Prepared {len(X)} samples with {len(feature_cols)} features")

    return X, y, metadata

def train_ml_pipeline(X, y, config):
    """Train ML pipeline with ensemble and uncertainty."""
    print("Training ML pipeline...")

    # Create data splitter
    splitter_config = SplitConfig(
        test_size=0.2,
        val_size=0.2,
        time_series_split=True,
        n_splits=3
    )
    splitter = DataSplitter(splitter_config)

    # Create ensemble model
    ensemble = StackingEnsemble(config)

    # Create training pipeline
    pipeline = MLTrainingPipeline(
        model=ensemble,
        splitter=splitter,
        output_dir=OUTPUT_DIR / "ml_training"
    )

    # Train the model
    training_results = pipeline.train(X, y)

    print("✓ ML training completed")
    return pipeline, training_results

# ===== EXECUTE ML TRAINING =====
print("Training Machine Learning Models...")
print("=" * 50)

ml_results = {}
training_histories = {}
trained_models = {}

# Train models for each horizon
for horizon_name in HORIZONS.keys():
    print(f"\n--- Training for horizon {horizon_name} ---")

    try:
        # Prepare data
        X, y, metadata = prepare_ml_data(combined_df, horizon_name)

        if X is None:
            print(f"Skipping {horizon_name} due to insufficient data")
            continue

        # Train model
        model, training_results = train_ml_pipeline(X, y, ENSEMBLE_CONFIG)

        # Store results
        ml_results[horizon_name] = {
            'X': X,
            'y': y,
            'metadata': metadata,
            'training_results': training_results
        }

        trained_models[horizon_name] = model
        training_histories[horizon_name] = training_results.get('history', {})

        print(f"✓ Model trained for {horizon_name}")

    except Exception as e:
        print(f"✗ Failed to train model for {horizon_name}: {e}")
        import traceback
        traceback.print_exc()

if not ml_results:
    raise ValueError("No ML models were successfully trained!")

print(f"\n✓ Successfully trained models for {len(ml_results)} horizons")

# Save training histories
with open(OUTPUT_DIR / "training_history.json", 'w') as f:
    json.dump(training_histories, f, indent=2, default=str)
print(f"✓ Training histories saved to {OUTPUT_DIR / 'training_history.json'}")

# Display training summary
print("\nTraining Summary:")
for horizon, results in ml_results.items():
    history = results['training_results']
    if 'best_score' in history:
        print(f"  {horizon}: Best score = {history['best_score']:.4f}")
    else:
        print(f"  {horizon}: Training completed")

## 8. Execute Hybrid Model

Combine physics and ML predictions to generate final soil moisture forecasts.

In [ ]:
# Hybrid Model Execution

def generate_hybrid_predictions(X, metadata, model, horizon_name):
    """Generate hybrid predictions by combining physics + ML."""
    print(f"Generating hybrid predictions for {horizon_name}...")

    # Get ML predictions (residuals)
    try:
        ml_predictions = model.predict(X)
    except Exception as e:
        print(f"  ✗ ML prediction failed: {e}")
        ml_predictions = np.zeros(len(X))

    # Get physics predictions
    physics_predictions = metadata[f'physics_pred_{horizon_name}'].values

    # Combine: physics + ML residual
    hybrid_predictions = physics_predictions + ml_predictions

    # Clip to valid soil moisture range
    hybrid_predictions = np.clip(hybrid_predictions, 0.0, 1.0)

    # Get uncertainty if available
    uncertainties = None
    if hasattr(model, 'predict_uncertainty'):
        try:
            uncertainties = model.predict_uncertainty(X)
        except:
            pass

    results_df = metadata.copy()
    results_df[f'ml_residual_{horizon_name}'] = ml_predictions
    results_df[f'hybrid_pred_{horizon_name}'] = hybrid_predictions
    results_df[f'observed_{horizon_name}'] = results_df['observed_soil_moisture']

    if uncertainties is not None:
        results_df[f'hybrid_uncertainty_{horizon_name}'] = uncertainties

    print(f"  ✓ Generated {len(results_df)} hybrid predictions")

    return results_df

# ===== EXECUTE HYBRID PREDICTIONS =====
print("Generating Hybrid Model Predictions...")
print("=" * 50)

hybrid_results = {}
paired_data = []

for horizon_name, results in ml_results.items():
    print(f"\n--- Generating predictions for {horizon_name} ---")

    try:
        X = results['X']
        metadata = results['metadata']
        model = trained_models[horizon_name]

        # Generate predictions
        horizon_results = generate_hybrid_predictions(X, metadata, model, horizon_name)
        hybrid_results[horizon_name] = horizon_results

        # Add to paired data for analysis
        paired_data.extend(horizon_results.to_dict('records'))

        print(f"✓ Hybrid predictions completed for {horizon_name}")

    except Exception as e:
        print(f"✗ Failed to generate predictions for {horizon_name}: {e}")
        import traceback
        traceback.print_exc()

if not hybrid_results:
    raise ValueError("No hybrid predictions were generated!")

# Combine all paired data
paired_df = pd.DataFrame(paired_data)
print(f"\n✓ Combined paired dataset: {len(paired_df)} rows")

# Save results
paired_df.to_csv(OUTPUT_DIR / "paired_obs_pred.csv", index=False)
print(f"✓ Paired data saved to {OUTPUT_DIR / 'paired_obs_pred.csv'}")

# Display prediction summary
print("\nPrediction Summary:")
for horizon in hybrid_results.keys():
    df = hybrid_results[horizon]
    obs_col = f'observed_{horizon}'
    pred_col = f'hybrid_pred_{horizon}'

    if obs_col in df.columns and pred_col in df.columns:
        valid_data = df[[obs_col, pred_col]].dropna()
        if len(valid_data) > 0:
            mae = np.mean(np.abs(valid_data[obs_col] - valid_data[pred_col]))
            rmse = np.sqrt(np.mean((valid_data[obs_col] - valid_data[pred_col])**2))
            print(f"  {horizon}: {len(valid_data)} points, MAE={mae:.4f}, RMSE={rmse:.4f}")

# Show sample predictions
print("\nSample Predictions:")
sample_df = paired_df.head(10)[['station_id', 'date', 'observed_0h', 'physics_pred_0h', 'ml_residual_0h', 'hybrid_pred_0h']]
sample_df

## 9. Perform Validation

Compute comprehensive validation metrics for physics, ML, and hybrid models.

In [ ]:
# Validation and Metrics Computation

def compute_validation_metrics(obs, pred, model_name="model"):
    """Compute comprehensive validation metrics."""
    if len(obs) == 0 or len(pred) == 0:
        return {}

    # Remove NaN values
    valid_mask = ~(np.isnan(obs) | np.isnan(pred))
    obs = obs[valid_mask]
    pred = pred[valid_mask]

    if len(obs) < 10:
        return {}

    # Basic metrics
    bias = np.mean(pred - obs)
    mae = np.mean(np.abs(pred - obs))
    rmse = np.sqrt(np.mean((pred - obs)**2))
    mse = np.mean((pred - obs)**2)

    # Relative metrics
    mape = np.mean(np.abs((obs - pred) / obs)) * 100 if np.all(obs > 0) else np.nan
    nse = 1 - (np.sum((obs - pred)**2) / np.sum((obs - np.mean(obs))**2)) if np.std(obs) > 0 else np.nan

    # Correlation metrics
    r = np.corrcoef(obs, pred)[0, 1] if len(obs) > 1 else 0
    r2 = r**2

    # Kling-Gupta Efficiency
    obs_mean = np.mean(obs)
    pred_mean = np.mean(pred)
    obs_std = np.std(obs)
    pred_std = np.std(pred)

    alpha = pred_std / obs_std if obs_std > 0 else 0
    beta = pred_mean / obs_mean if obs_mean > 0 else 0

    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

    # Unbiased RMSE
    obs_anom = obs - np.mean(obs)
    pred_anom = pred - np.mean(pred)
    ubrmse = np.sqrt(np.mean((obs_anom - pred_anom)**2))

    return {
        'model': model_name,
        'n_points': len(obs),
        'bias': bias,
        'mae': mae,
        'rmse': rmse,
        'mse': mse,
        'mape': mape,
        'nse': nse,
        'r': r,
        'r2': r2,
        'kge': kge,
        'ubrmse': ubrmse,
        'obs_mean': np.mean(obs),
        'pred_mean': np.mean(pred),
        'obs_std': np.std(obs),
        'pred_std': np.std(pred)
    }

# ===== EXECUTE VALIDATION =====
print("Computing Validation Metrics...")
print("=" * 50)

validation_results = []

for horizon_name, results_df in hybrid_results.items():
    print(f"\n--- Validation for {horizon_name} ---")

    obs_col = f'observed_{horizon_name}'
    physics_col = f'physics_pred_{horizon_name}'
    hybrid_col = f'hybrid_pred_{horizon_name}'

    # Validate physics model
    if physics_col in results_df.columns:
        obs = results_df[obs_col].values
        physics_pred = results_df[physics_col].values

        physics_metrics = compute_validation_metrics(obs, physics_pred, f"physics_{horizon_name}")
        if physics_metrics:
            validation_results.append(physics_metrics)
            print(f"  Physics - RMSE: {physics_metrics['rmse']:.4f}, KGE: {physics_metrics['kge']:.4f}")

    # Validate hybrid model
    if hybrid_col in results_df.columns:
        obs = results_df[obs_col].values
        hybrid_pred = results_df[hybrid_col].values

        hybrid_metrics = compute_validation_metrics(obs, hybrid_pred, f"hybrid_{horizon_name}")
        if hybrid_metrics:
            validation_results.append(hybrid_metrics)
            print(f"  Hybrid  - RMSE: {hybrid_metrics['rmse']:.4f}, KGE: {hybrid_metrics['kge']:.4f}")

# Convert to DataFrame
validation_df = pd.DataFrame(validation_results)
print(f"\n✓ Computed metrics for {len(validation_results)} model-horizon combinations")

# Save validation results
validation_df.to_csv(OUTPUT_DIR / "ml_validation_results.csv", index=False)
print(f"✓ Validation results saved to {OUTPUT_DIR / 'ml_validation_results.csv'}")

# Display validation summary
print("\nValidation Summary:")
summary_table = validation_df.pivot_table(
    index='model',
    values=['rmse', 'mae', 'kge', 'r2', 'n_points'],
    aggfunc='mean'
).round(4)
summary_table

## 10. Generate Reports and Plots

Create validation reports, scatter plots, and performance visualizations.

In [ ]:
# Reports and Plots Generation

def create_scatter_plots(validation_df, paired_df):
    """Create scatter plots comparing observed vs predicted values."""
    print("Creating scatter plots...")

    # Set up the plotting style
    plt.style.use('default')
    sns.set_palette("husl")

    horizons = list(HORIZONS.keys())
    models = ['physics', 'hybrid']

    fig, axes = plt.subplots(len(horizons), len(models),
                            figsize=(4*len(models), 3*len(horizons)),
                            squeeze=False)

    for i, horizon in enumerate(horizons):
        for j, model_type in enumerate(models):
            ax = axes[i, j]

            # Get data for this horizon and model
            model_col = f"{model_type}_{horizon}"
            obs_col = f"observed_{horizon}"
            pred_col = f"{model_type}_pred_{horizon}"

            if pred_col not in paired_df.columns:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
                continue

            # Filter valid data
            plot_data = paired_df[[obs_col, pred_col]].dropna()

            if len(plot_data) < 10:
                ax.text(0.5, 0.5, 'Insufficient Data', ha='center', va='center', transform=ax.transAxes)
                continue

            obs = plot_data[obs_col]
            pred = plot_data[pred_col]

            # Create scatter plot
            ax.scatter(obs, pred, alpha=0.6, s=20, edgecolors='none')

            # Add 1:1 line
            min_val = min(obs.min(), pred.min())
            max_val = max(obs.max(), pred.max())
            ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7, label='1:1 line')

            # Add metrics text
            metrics = compute_validation_metrics(obs, pred, model_col)
            if metrics:
                text = '.4f'
                ax.text(0.05, 0.95, text, transform=ax.transAxes,
                       verticalalignment='top', fontsize=9,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

            ax.set_xlabel('Observed Soil Moisture (m³/m³)')
            ax.set_ylabel('Predicted Soil Moisture (m³/m³)')
            ax.set_title(f'{model_type.title()} Model - {horizon}')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(min_val, max_val)
            ax.set_ylim(min_val, max_val)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "scatter_plots.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✓ Scatter plots saved to {OUTPUT_DIR / 'scatter_plots.png'}")

def create_performance_summary(validation_df):
    """Create performance summary plots."""
    print("Creating performance summary plots...")

    # Filter to hybrid models for summary
    hybrid_data = validation_df[validation_df['model'].str.startswith('hybrid_')].copy()

    if len(hybrid_data) == 0:
        print("No hybrid model data for summary plots")
        return

    # Extract horizon from model name
    hybrid_data['horizon'] = hybrid_data['model'].str.replace('hybrid_', '')

    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # RMSE by horizon
    axes[0,0].bar(hybrid_data['horizon'], hybrid_data['rmse'])
    axes[0,0].set_title('RMSE by Forecast Horizon')
    axes[0,0].set_ylabel('RMSE (m³/m³)')
    axes[0,0].grid(True, alpha=0.3)

    # KGE by horizon
    axes[0,1].bar(hybrid_data['horizon'], hybrid_data['kge'])
    axes[0,1].set_title('KGE by Forecast Horizon')
    axes[0,1].set_ylabel('KGE')
    axes[0,1].axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='Good (0.5)')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)

    # R² by horizon
    axes[1,0].bar(hybrid_data['horizon'], hybrid_data['r2'])
    axes[1,0].set_title('R² by Forecast Horizon')
    axes[1,0].set_ylabel('R²')
    axes[1,0].grid(True, alpha=0.3)

    # Sample size by horizon
    axes[1,1].bar(hybrid_data['horizon'], hybrid_data['n_points'])
    axes[1,1].set_title('Sample Size by Forecast Horizon')
    axes[1,1].set_ylabel('Number of Points')
    axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "performance_summary.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✓ Performance summary saved to {OUTPUT_DIR / 'performance_summary.png'}")

def generate_validation_report(validation_df, paired_df):
    """Generate comprehensive validation report."""
    print("Generating validation report...")

    report_path = OUTPUT_DIR / "validation_report.txt"

    with open(report_path, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("SMPS HYBRID PHYSICS + ML VALIDATION REPORT\n")
        f.write("=" * 80 + "\n\n")

        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Date range: {START_DATE} to {END_DATE}\n")
        f.write(f"Stations processed: {paired_df['station_id'].nunique()}\n")
        f.write(f"Total predictions: {len(paired_df)}\n\n")

        f.write("MODEL PERFORMANCE SUMMARY\n")
        f.write("-" * 40 + "\n")

        # Group by model type
        for model_type in ['physics', 'hybrid']:
            f.write(f"\n{model_type.upper()} MODEL:\n")

            model_data = validation_df[validation_df['model'].str.startswith(model_type)]
            if len(model_data) > 0:
                summary = model_data[['rmse', 'mae', 'kge', 'r2']].mean()
                f.write(f"  Mean RMSE: {summary['rmse']:.4f}\n")
                f.write(f"  Mean MAE:  {summary['mae']:.4f}\n")
                f.write(f"  Mean KGE:  {summary['kge']:.4f}\n")
                f.write(f"  Mean R²:   {summary['r2']:.4f}\n")
            else:
                f.write("  No data available\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("DETAILED METRICS BY HORIZON\n")
        f.write("=" * 80 + "\n")

        for horizon in HORIZONS.keys():
            f.write(f"\n{horizon} Forecast Horizon:\n")
            f.write("-" * 20 + "\n")

            horizon_data = validation_df[validation_df['model'].str.contains(horizon)]
            if len(horizon_data) > 0:
                for _, row in horizon_data.iterrows():
                    model_name = row['model'].replace(f'_{horizon}', '')
                    f.write("12s")
            else:
                f.write("  No data available\n")

    print(f"✓ Validation report saved to {report_path}")

# ===== EXECUTE REPORTS AND PLOTS =====
print("Generating Reports and Plots...")
print("=" * 50)

try:
    # Create scatter plots
    create_scatter_plots(validation_df, paired_df)
except Exception as e:
    print(f"⚠ Failed to create scatter plots: {e}")

try:
    # Create performance summary
    create_performance_summary(validation_df)
except Exception as e:
    print(f"⚠ Failed to create performance summary: {e}")

try:
    # Generate validation report
    generate_validation_report(validation_df, paired_df)
except Exception as e:
    print(f"⚠ Failed to generate validation report: {e}")

print("\n✓ Report generation completed!")
print(f"All results saved to: {OUTPUT_DIR}")

# Final summary
print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)
print(f"✓ Processed {paired_df['station_id'].nunique()} stations")
print(f"✓ Generated {len(paired_df)} predictions across {len(HORIZONS)} horizons")
print(f"✓ Results saved to {OUTPUT_DIR}")
print("\nKey output files:")
print(f"  - combined_features.csv: Complete feature dataset")
print(f"  - paired_obs_pred.csv: Predictions vs observations")
print(f"  - ml_validation_results.csv: Performance metrics")
print(f"  - training_history.json: ML training curves")
print(f"  - scatter_plots.png: Scatter plot visualizations")
print(f"  - performance_summary.png: Performance summary plots")
print(f"  - validation_report.txt: Comprehensive report")
print("=" * 80)